# 01 — Exploratory Data Analysis
**Energy Demand Forecasting** | OPSD Germany dataset

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates, seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from src.data_loader import build_dataset, load_processed
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
%matplotlib inline

## Download & preview data

In [ ]:
df = build_dataset()
df.head()

In [ ]:
print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0]}  →  {df.index[-1]}')
print(f'\nMissing values:\n{df.isna().sum()}')
print(f'\nLoad statistics (MW):')
print(df['load_mw'].describe().round(1))

## Full time series

In [ ]:
fig, ax = plt.subplots(figsize=(16,4))
ax.plot(df.index, df['load_mw'], lw=0.3, color='#2563EB', alpha=0.85)
ax.set_title('Germany hourly electricity demand (2006–2017)', fontsize=13)
ax.set_ylabel('Load (MW)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()

## Seasonality: hour / day / month

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,4))
df.groupby(df.index.hour)['load_mw'].mean().plot.bar(ax=axes[0], color='#2563EB', alpha=0.8)
axes[0].set_title('By hour of day'); axes[0].set_xlabel('Hour')

days=['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
df.groupby(df.index.dayofweek)['load_mw'].mean().plot.bar(ax=axes[1], color='#16A34A', alpha=0.8)
axes[1].set_title('By day of week'); axes[1].set_xticklabels(days, rotation=0)

months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df.groupby(df.index.month)['load_mw'].mean().plot.bar(ax=axes[2], color='#D97706', alpha=0.8)
axes[2].set_title('By month'); axes[2].set_xticklabels(months, rotation=45)
plt.tight_layout()

## Seasonal decomposition (additive)

In [ ]:
subset = df['load_mw']['2012':'2013']
result = seasonal_decompose(subset.dropna(), model='additive', period=8760)
fig, axes = plt.subplots(4,1,figsize=(16,10),sharex=True)
for ax,(title,comp,col) in zip(axes,[
    ('Observed', result.observed,'#2563EB'),
    ('Trend',    result.trend,   '#16A34A'),
    ('Seasonal', result.seasonal,'#D97706'),
    ('Residual', result.resid,   'gray'),
]):
    ax.plot(comp, color=col, lw=0.5); ax.set_title(title); ax.set_ylabel('MW')
plt.suptitle('Seasonal decomposition 2012–2013', y=1.01); plt.tight_layout()

## ACF / PACF

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(14,7))
plot_acf(df['load_mw'].dropna(), lags=72, ax=axes[0], zero=False)
axes[0].set_title('ACF — up to 72 lags')
plot_pacf(df['load_mw'].dropna(), lags=72, ax=axes[1], zero=False, method='ywm')
axes[1].set_title('PACF — up to 72 lags')
plt.tight_layout()

## ADF Stationarity Test

In [ ]:
from statsmodels.tsa.stattools import adfuller
r = adfuller(df['load_mw'].dropna())
print(f'ADF Statistic : {r[0]:.4f}')
print(f'p-value       : {r[1]:.4f}')
print(f'Stationary?   : {"Yes" if r[1] < 0.05 else "No"}')